In [66]:
import pandas as pd
import numpy as np
from pathlib import Path
import csv
from datetime import datetime, timedelta

def extract_pos_coordinates(filepath):
    """
    Extract specified columns from a position file.

    Parameters:
    - filepath: Path to the input file
    - columns_to_extract: List of column indices to extract (0-based). 
                         If None, extracts first 4 columns by default.
    - output_csv: Optional path to save results as CSV
    - column_names: Optional list of custom column names (same length as columns_to_extract)
    """
    output_csv = None
    if filepath[-9] == 'K':
        columns_to_extract = [0, 1, 2, 3]
        column_names=['GPST', 'x_y', 'y_y', 'z_y']
    else:
        columns_to_extract = [0,1,2,3,5,6,7,8]
        column_names=['GPST', 'x_x', 'y_x', 'z_x','ns','sdx(m)','sdy(m)','sdz(m)']

    extracted_rows = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip metadata and comments
            if not line or line.startswith('%'):
                continue

            parts = [p.strip() for p in line.split(',')]

            try:
                extracted_values = []
                for col_idx in columns_to_extract:
                    if col_idx < len(parts):
                        try:
                            extracted_values.append(float(parts[col_idx]))
                        except ValueError:
                            extracted_values.append(parts[col_idx])
                    else:
                        extracted_values.append(None)
                extracted_rows.append(extracted_values)

            except (IndexError, ValueError) as e:
                print(f"Warning: Skipping line due to error: {e}")
                print(f"Line content: {line}")
                continue

    # Write to CSV (with custom or default headers)
    if output_csv:
        if column_names is None:
            column_names = [f'Column_{idx}' for idx in columns_to_extract]
        elif len(column_names) != len(columns_to_extract):
            raise ValueError("column_names length must match columns_to_extract length")

        with open(output_csv, 'w', nNSline='') as f_out:
            writer = csv.writer(f_out)
            writer.writerow(column_names)
            writer.writerows(extracted_rows)

    return pd.DataFrame(extracted_rows, columns=column_names)

def gpst_to_datetime(week, tow):
    """
    Converts GPS Week and Time of Week (TOW) to a formatted string.
    GPS Time starts on Jan 6, 1980.
    """
    gps_epoch = datetime(1980, 1, 6, 0, 0, 0)
    
    # Calculate the exact time
    elapsed = timedelta(weeks=float(week), seconds=float(tow))
    current_time = gps_epoch + elapsed
    
    # Format to match your pos file: 'YYYY/MM/DD HH:MM:SS.ss'
    # We slice [:-4] to trim microsecond precision down to 2 decimals if needed
    return current_time.strftime('%Y/%m/%d %H:%M:%S.%f')[:-4]

def parse_rtk_stat_file(stat_file_path):
    data = []
    
    with open(stat_file_path, 'r') as f:
        current_tow = None
        current_week = None
        
        snr_values = []
        res_values = []
        
        for line in f:
            if not line.startswith("$SAT"): continue
            
            parts = line.split(',')
            
            try:
                week = int(parts[1])
                tow = float(parts[2])
            except ValueError:
                continue

            # If we moved to a nNS timestamp, save the previous batch
            if current_tow is not None and tow != current_tow:
                if snr_values:
                    # Convert the time for the PREVIOUS batch
                    time_str = gpst_to_datetime(current_week, current_tow)
                    
                    data.append({
                        'GPST': time_str,  # Matches your POS file column name
                        'avg_snr': np.mean(snr_values),
                        'min_snr': np.min(snr_values),
                        'max_residual': np.max(np.abs(res_values))
                    })
                snr_values = []
                res_values = []
            
            current_week = week
            current_tow = tow
            
            # --- EXTRACT FEATURES ---
            try:
                snr = float(parts[10]) # Column 10: SNR
                res = float(parts[7])  # Column 7: Residual
                
                if snr > 0:
                    snr_values.append(snr)
                    res_values.append(res)
            except (ValueError, IndexError):
                continue

    # Don't forget the very last batch
    if current_tow is not None and snr_values:
        time_str = gpst_to_datetime(current_week, current_tow)
        data.append({
            'GPST': time_str,
            'avg_snr': np.mean(snr_values),
            'min_snr': np.min(snr_values),
            'max_residual': np.max(np.abs(res_values))
        })
    

    df_snr = pd.DataFrame(data)
    df_snr['GPST'] = pd.to_datetime(df_snr['GPST']).dt.round('s')

    return df_snr

# --- HOW TO USE ---
# 1. Parse the stat file
df_snr = parse_rtk_stat_file(r'F:\zizo\RTKCorrection\src\research\data\alll_constellation\10_NS_Morning_S_ALL.pos.stat')

# 2. Load your main POS file (assuming it has a 'GPST' or 'tow' column to match)
df_pos_S = extract_pos_coordinates(r'F:\zizo\RTKCorrection\src\research\data\alll_constellation\10_NS_Morning_S_ALL.pos')
df_pos_K = extract_pos_coordinates(r'F:\zizo\RTKCorrection\src\research\data\alll_constellation\10_NS_Morning_K_ALL.pos')

# (You might need to calculate 'tow' from the GPST timestamp in df_pos to merge them)

print(df_snr.head())

                 GPST    avg_snr  min_snr  max_residual
0 2023-12-10 02:06:08  29.909091     22.0        5.9485
1 2023-12-10 02:06:09  30.600000     23.0        5.9296
2 2023-12-10 02:06:10  30.454545     23.0        7.5984
3 2023-12-10 02:06:11  31.200000     23.0        5.4988
4 2023-12-10 02:06:12  31.300000     23.0        7.7423


In [67]:
df_pos_S.head()

,GPST,x_x,y_x,z_x,ns,sdx(m),sdy(m),sdz(m)
0,2023/12/10 02:06:08.00,459274.6746,5.634064e+06,2.946975e+06,11.0,3.6350,14.8532,5.8574
1,2023/12/10 02:06:09.00,459280.3367,5.634075e+06,2.946973e+06,10.0,5.9052,23.7337,5.8577
2,2023/12/10 02:06:10.00,459278.3289,5.634070e+06,2.946974e+06,11.0,5.9013,23.6777,5.7342
3,2023/12/10 02:06:11.00,459284.0392,5.634091e+06,2.946978e+06,10.0,6.6479,25.9971,5.8282
4,2023/12/10 02:06:12.00,459282.9088,5.634087e+06,2.946978e+06,10.0,6.6473,25.9862,5.8282


In [68]:
df_pos_K.head()

,GPST,x_y,y_y,z_y
0,2023/12/10 02:07:00.00,459299.3515,5.634059e+06,2.946956e+06
1,2023/12/10 02:07:03.00,459301.9326,5.634057e+06,2.946954e+06
2,2023/12/10 02:07:04.00,459301.8875,5.634057e+06,2.946954e+06
3,2023/12/10 02:07:05.00,459248.7685,5.634077e+06,2.946982e+06
4,2023/12/10 02:07:10.00,459252.4164,5.634076e+06,2.946980e+06


In [69]:
len(df_snr), len(df_pos_S), len(df_pos_K)

(14231, 14231, 13494)

In [70]:
df_pos_S['GPST'] = pd.to_datetime(df_pos_S['GPST']).dt.round('s')
df_snr['GPST'] = pd.to_datetime(df_snr['GPST']).dt.round('s')
df = pd.merge(df_snr, df_pos_S, on='GPST', how='inner')
df.to_csv(r'F:\zizo\RTKCorrection\src\research\data\alll_constellation\10_NS_Morning_S_ALL.csv', index=False)
len(df)

14231

In [71]:
df_pos_K.to_csv(r'F:\zizo\RTKCorrection\src\research\data\alll_constellation\10_NS_Morning_K_ALL.csv', index=False)


In [82]:
df1= pd.read_csv(r'F:\zizo\RTKCorrection\src\research\data\alll_constellation\10_NS_Morning_K_ALL.csv')
df1['GPST'] = pd.to_datetime(df1['GPST']).dt.round('s')
df1.to_csv(r'F:\zizo\RTKCorrection\src\research\data\alll_constellation\10_NS_Morning_K_ALL.csv', index=False)